# Corte 3, Punto 2: flujo continuo y ventanas temporales

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T2.1 a T2.6 (simulación, micro-lotes, lectura en streaming, ventanas, llegada tardía y validación contra el cálculo por lotes).

Entrada: gold.

_Por hacer._

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
import time
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    SILVER_PARQUET,
    GOLD_PARQUET,
    STREAMING_LOTES,
    STREAMING_ENTRADA,
    STREAMING_TARDIOS,
    STREAMING_CHECKPOINT,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark
from pamd.calidad import marcar_atipicos, unificar_nit_entidad

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T2.1 Diseño de la simulación

Aprobado por Gabriel Rodríguez el 15-nov-2026 (criterio 3 de T2.1); se informa a Jonatan Palomares. Todos los parámetros están en la celda siguiente y se pueden cambiar sin tocar el resto del notebook.

**Qué se simula.** SECOP II no publica los contratos como un flujo continuo, así que el flujo se simula: gold se parte en archivos (micro-lotes) que llegan uno a uno a una carpeta de MinIO, y Spark Structured Streaming los procesa a medida que aparecen.

| Aspecto | Decisión |
|---|---|
| Origen | gold con las marcas de `src/pamd/calidad.py` (`marcar_atipicos(unificar_nit_entidad(gold), silver)`), solo con las columnas que usa el flujo |
| Lotes | `NUM_LOTES` = 20 lotes regulares en orden de `fecha_adjudicacion`, cortados por meses completos (un mes nunca queda partido entre dos lotes) y con un número parecido de contratos |
| Carpetas | los lotes se preparan en `streaming/lotes/` y se copian uno a uno a `streaming/entrada/`, que es la carpeta que lee el flujo |
| Ritmo | llega un lote cada `PAUSA_SEGUNDOS` = 5 s; el flujo lee con `maxFilesPerTrigger = 1`, un lote por disparo |
| Qué es tardío | un contrato que llega en un lote posterior a otro que ya traía meses más recientes que el suyo |
| Tardíos inyectados | dos lotes extra con el `PORCENTAJE_TARDIO` = 20 % de los contratos de un mes, elegidos con un hash de `id_adjudicacion` y la `SEMILLA` = 13 (misma semilla, mismos lotes). **Dentro de la tolerancia:** penúltimo mes del lote 10, llega justo después del lote 10. **Fuera de la tolerancia:** primer mes del lote 5, llega después del lote 15 |
| Ventana | mes calendario exacto: se agrupa por el inicio del mes (`date_trunc`) y por el departamento de la entidad, con `withWatermark` sobre ese inicio de mes. La versión anual se obtiene sumando los meses de cada año |
| Tolerancia | `TOLERANCIA` = 75 días antes del mes más reciente visto: se aceptan los dos meses anteriores. 75 días nunca cae en un día 1, así que no hay casos en el límite |
| Fuera de la tolerancia | Spark descarta el registro en la agregación; el flujo lo guarda en `streaming/tardios/` y su ventana se corrige sumando esa tabla, así que ninguno se pierde en silencio |
| Medidas | las mismas de las tablas de T1.4: contratos, contratos atípicos, contratos con valor cero, `valor_total` (sin atípicos) y `valor_atipicos` |
| Salida | modo `update` con `foreachBatch`: en cada disparo llegan solo las ventanas que cambiaron. En T2 se guardan en una tabla en memoria; el destino idempotente en MinIO es la T3.2 |
| Punto de control | `streaming/checkpoint/ventanas` (T3.1) |

**Por qué mes calendario y no `window()` de duración fija:** el mes exacto coincide con `anio_mes` de las tablas resumen y del dashboard, y como el inicio del mes es una marca de tiempo sigue admitiendo marca de agua; una ventana fija de 30 días no coincide con los meses.

In [2]:
# T2.1 · Parámetros de la simulación (se cambian aquí; el resto del notebook los usa)
NUM_LOTES = 20                         # lotes regulares, en orden de fecha_adjudicacion y por meses completos
PAUSA_SEGUNDOS = 5                     # espera entre la llegada de un lote y el siguiente
SEMILLA = 13                           # semilla del hash que elige los contratos tardíos
PORCENTAJE_TARDIO = 20                 # % de contratos del mes elegido que se retienen y llegan tarde
TOLERANCIA = "75 days"                 # marca de agua: se aceptan los dos meses anteriores al más reciente visto
DISPARO = "2 seconds"                  # cada cuánto revisa el flujo si llegó un archivo nuevo
PARTICIONES_ESTADO = 4                 # particiones del estado de la agregación (quedan fijadas en el checkpoint)
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"  # misma categoría que las tablas de T1.4

# Lotes tardíos: lote regular del que sale el mes, qué mes de ese lote (0 = el primero, -2 = el penúltimo) y tras qué lote llega
TARDIOS = {
    "tardio_dentro": {"lote_origen": 10, "mes_en_lote": -2, "llega_tras_lote": 10},  # dentro de la tolerancia
    "tardio_fuera": {"lote_origen": 5, "mes_en_lote": 0, "llega_tras_lote": 15},     # fuera de la tolerancia
}

assert all(1 <= t["lote_origen"] <= t["llega_tras_lote"] <= NUM_LOTES for t in TARDIOS.values()), \
    "Un lote tardío debe llegar después de su lote de origen"
print(f"{NUM_LOTES} lotes regulares + {len(TARDIOS)} tardíos = {NUM_LOTES + len(TARDIOS)} archivos; "
      f"un lote cada {PAUSA_SEGUNDOS} s; tolerancia de {TOLERANCIA}")

20 lotes regulares + 2 tardíos = 22 archivos; un lote cada 5 s; tolerancia de 75 days
